In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
class Value:
    def __init__(self, data, _children =(), _op = '' ):
        self.data = data
        self.grad = 0
        self._prev = set(_children)
        self._backward = lambda : None
        self._op = _op
    
    def __repr__(self):
        return f"Value(data={self.data})"
    
    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1 * out.grad
            other.grad += 1 * out.grad
            
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def tanh(self):
        n = self.data
        t = (math.exp(2*n) - 1) / (math.exp(2*n) + 1)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out
    
    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)

                for child in v._prev:
                    build_topo(child)

                topo.append(v)

        build_topo(self)

        self.grad = 1.0 # Because this is the output node and we will start the back propagation from here.

        for node in reversed(topo):
            node._backward()

In [3]:
# Inputs x1, x2

x1 = Value(2.0)
x2 = Value(0)

# Weights w1, w2

w1 = Value(-3.0)
w2 = Value(1.0)

# Bias of the neuron

b = Value(6.8813735870195432)

x1w1 = x1 * w1
x2w2 = x2 * w2
x1w1x2w2 = x1w1 + x2w2
n = x1w1x2w2 + b

# Output
o = n.tanh()


### Now we have an issue where we cannot compute the following

In [4]:
a = Value(3.0)
a + 3

AttributeError: 'int' object has no attribute 'data'

### This will always throw an error because when we define the add function, we have defined it as (self, other) and it considers the integer 3 as other and 3 has no data attribute to add there this will always throw an error and we now solve that

In [8]:
class Value:
    def __init__(self, data, _children =(), _op = '' ):
        self.data = data
        self.grad = 0
        self._prev = set(_children)
        self._backward = lambda : None
        self._op = _op
    
    def __repr__(self):
        return f"Value(data={self.data})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1 * out.grad
            other.grad += 1 * out.grad
            
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def tanh(self):
        n = self.data
        t = (math.exp(2*n) - 1) / (math.exp(2*n) + 1)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out
    
    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)

                for child in v._prev:
                    build_topo(child)

                topo.append(v)

        build_topo(self)

        self.grad = 1.0 # Because this is the output node and we will start the back propagation from here.

        for node in reversed(topo):
            node._backward()

In [9]:
a = Value(3.0)
a + 3

Value(data=6.0)

In [10]:
a = Value(3.0)
a + 3.3

Value(data=6.3)

In [11]:
a = Value(3.0)
a * 3.3

Value(data=9.899999999999999)

In [12]:
2 * a

TypeError: unsupported operand type(s) for *: 'int' and 'Value'

### Now we are able to solve it for all the values but there's one catch lol, we can do a * 2 but we cannot do 2 * a because python calls the .__mul__ function and it cannot perform the mul function on a value therefore it will give an error. 

### For example: 

a * 2 calls a.__mul__(2) "a" is a value and we have defined the mul function for a value explicitly but when we do 2 * a, then we dont know 2.__mul(a) because the integers dont know how to perform multiplication on a value.

In [17]:
class Value:
    def __init__(self, data, _children =(), _op = '' ):
        self.data = data
        self.grad = 0
        self._prev = set(_children)
        self._backward = lambda : None
        self._op = _op
    
    def __repr__(self):
        return f"Value(data={self.data})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1 * out.grad
            other.grad += 1 * out.grad
            
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def __rmul__(self, other):
        return self * other
    
    def __radd__(self, other):  
        return self + other
    
    def tanh(self):
        n = self.data
        t = (math.exp(2*n) - 1) / (math.exp(2*n) + 1)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out
    
    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)

                for child in v._prev:
                    build_topo(child)

                topo.append(v)

        build_topo(self)

        self.grad = 1.0 # Because this is the output node and we will start the back propagation from here.

        for node in reversed(topo):
            node._backward()

In [18]:
a = Value(3.0)
a * 3.3

Value(data=9.899999999999999)

In [19]:
2 * a

Value(data=6.0)

In [20]:
2 + a

Value(data=5.0)